# Exploratory Patient Location Mapping
This notebooks explores the patient records geographic distribtuion in Kansas. It performs some data wrangling and cleaning which is then used with Folium to display patient locations and clustered map markers.

Inputs: `departments.csv`, `patients.csv`, and  `tigercensuscodes.csv` (restricted ASA Datafest data, not included)



In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import folium

current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        folder for folder in (current_dir, *current_dir.parents)
        if (folder / 'data').is_dir()
        and (folder / 'notebooks').is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Project folder not found.'
    )

DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'private' / 'raw'
PROCESSED_DIR = DATA_DIR / 'private' / 'processed'
GEO_DIR = DATA_DIR / 'geospatial'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Raw data folder:', RAW_DIR)

In [ ]:
dept_df=pd.read_csv(RAW_DIR / 'departments.csv')
dept_df.head()

In [ ]:
dept_df.shape

In [ ]:
dept_df.isna().sum()

In [ ]:
dept_df.duplicated().sum()

In [ ]:
dept_df.info()

In [ ]:
dept_df=dept_df.dropna(subset=['CensusTract'])
dept_df.head()

In [ ]:
dept_df.shape

In [ ]:
patients_df=pd.read_csv(RAW_DIR / 'patients.csv')
patients_df.head()

In [ ]:
tiger_df=pd.read_csv(RAW_DIR / 'tigercensuscodes.csv')
tiger_df['GEOID']=tiger_df['GEOID'].astype('str')
#tiger_df.head()
#tiger_df.info()

In [ ]:
merged_df=pd.merge(patients_df,tiger_df,left_on='CensusBlockGroupFipsCode',right_on='GEOID')
merged_df.head()

In [ ]:
kansas_lat=39.01
kansas_lon=-98.48
m=folium.Map(location=[kansas_lat, kansas_lon], tiles="CartoDB positron")

In [ ]:
merged_df_filtered=merged_df.dropna(subset=['GEOID','CENTLAT','CENTLON'])
merged_df_filtered.head()

In [ ]:
# Select up to 100 records for visualization
MAX_ROWS = 100

plot_df = merged_df_filtered.sample(
    n=min(MAX_ROWS, len(merged_df_filtered)),
    random_state=42
)

print(f"Displaying {len(plot_df)} of {len(merged_df_filtered)} records")

In [ ]:
#idea for each block display majority info/cluster
for patient in plot_df.iterrows():
    folium.Marker(
        location=[patient[1]['CENTLAT'],patient[1]['CENTLON']]
    ).add_to(m)

In [ ]:
def getCommonLocations(df):
    valid = df[
        (df['GEOID'].notna()) &
        df['CENTLAT'].notna() &
        df['CENTLON'].notna()
    ].copy()
    
    valid['lat_r']=valid['CENTLAT'].round(2)
    valid['lon_r']=valid['CENTLON'].round(2)

    groups = valid.groupby(['lat_r', 'lon_r']).agg({
        'CENTLAT': 'first',
        'CENTLON': 'first',
    }).reset_index()

    groups.columns = ['lat_r', 'lon_r','CENTLAT','CENTLON']
    return groups.to_dict("records")

locations = getCommonLocations(plot_df)
locations

In [ ]:
for loc in locations:
    folium.CircleMarker(
        [loc['lat_r'], loc['lon_r']],
    ).add_to(m)

In [ ]:
import webbrowser

map_path = OUTPUT_DIR / "patient_locations_sample.html"

m.save(str(map_path))

webbrowser.open(map_path.resolve().as_uri())